# Data Cleaning
This ipynb file is to do the data cleaning for the file (world-education-dataset.csv)

### 1. Load the Dataset

In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset
file_path = "world-education-data.csv"
df = pd.read_csv(file_path)

# Display the first 5 rows to confirm it loaded correctly
print("--- FIRST 5 ROWS OF THE DATASET ---")
display(df.head())

--- FIRST 5 ROWS OF THE DATASET ---


,country,country_code,year,gov_exp_pct_gdp,lit_rate_adult_pct,pri_comp_rate_pct,pupil_teacher_primary,pupil_teacher_secondary,school_enrol_primary_pct,school_enrol_secondary_pct,school_enrol_tertiary_pct
0,Afghanistan,AFG,1999,NaN,NaN,NaN,33.18571,NaN,27.298849,NaN,NaN
1,Afghanistan,AFG,2000,NaN,NaN,NaN,NaN,NaN,22.162991,NaN,NaN
2,Afghanistan,AFG,2001,NaN,NaN,NaN,NaN,NaN,22.908590,14.47151,NaN
3,Afghanistan,AFG,2002,NaN,NaN,NaN,NaN,NaN,75.959747,NaN,NaN
4,Afghanistan,AFG,2003,NaN,NaN,NaN,NaN,NaN,96.553680,14.07805,1.38107


### 2. Check for Missing Values

In [2]:
# 2. Dataset Checking
print("--- DATASET SHAPE (Rows, Columns) ---")
print(df.shape)  # Should verify it has over 3,000 rows and 11 columns

print("\n--- DATA TYPES AND COMPLIANCE CHECK ---")
print(df.info())

print("\n--- MISSING VALUES PER ATTRIBUTE ---")
missing_values = df.isnull().sum()
print(missing_values)

print("\n--- SUMMARY STATISTICS FOR NUMERICAL COLUMNS ---")
display(df.describe())

--- DATASET SHAPE (Rows, Columns) ---
(5892, 11)

--- DATA TYPES AND COMPLIANCE CHECK ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5892 entries, 0 to 5891
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     5892 non-null   object 
 1   country_code                5892 non-null   object 
 2   year                        5892 non-null   int64  
 3   gov_exp_pct_gdp             4499 non-null   float64
 4   lit_rate_adult_pct          1877 non-null   float64
 5   pri_comp_rate_pct           4440 non-null   float64
 6   pupil_teacher_primary       3676 non-null   float64
 7   pupil_teacher_secondary     3017 non-null   float64
 8   school_enrol_primary_pct    5352 non-null   float64
 9   school_enrol_secondary_pct  4745 non-null   float64
 10  school_enrol_tertiary_pct   4392 non-null   float64
dtypes: float64(8), int64(1), object(2)
memory usage: 506.5+ K

,year,gov_exp_pct_gdp,lit_rate_adult_pct,pri_comp_rate_pct,pupil_teacher_primary,pupil_teacher_secondary,school_enrol_primary_pct,school_enrol_secondary_pct,school_enrol_tertiary_pct
count,5892.000000,4499.000000,1877.000000,4440.000000,3676.000000,3017.000000,5352.000000,4745.000000,4392.000000
mean,2010.921419,4.320129,79.483333,87.776740,25.344398,17.560340,101.525234,78.939810,36.533796
std,7.119808,1.736997,17.186877,17.857748,12.780357,7.465528,13.029901,28.350998,26.960123
min,1999.000000,0.242600,14.000000,14.411250,5.360520,4.979320,8.447979,3.293810,0.117370
25%,2005.000000,3.180390,65.984039,80.308426,15.888230,11.983680,97.281084,59.364799,12.605780
50%,2011.000000,4.101967,83.915154,94.604504,22.172125,16.224470,101.556335,85.707581,30.962285
75%,2017.000000,5.163850,94.215561,99.081194,32.569205,21.583330,106.822365,99.230003,57.325488
max,2023.000000,15.863470,100.000000,156.167175,100.236490,80.052320,257.434204,194.460022,166.665649


### 3. Dataset Cleaning

In [3]:
# 3. Dataset Cleaning

# Create a copy of the dataframe to keep the raw data safe
df_clean = df.copy()

# Step A: Drop rows where critical identifying information is missing
df_clean = df_clean.dropna(subset=['country', 'year'])

# Step B: Handle missing values in numerical attributes
# We group by country and use forward-fill (ffill) and backward-fill (bfill).
# This fills a missing year's data using the closest available year from the SAME country.
numerical_cols = [
    'gov_exp_pct_gdp', 'lit_rate_adult_pct', 'pri_comp_rate_pct', 
    'pupil_teacher_primary', 'pupil_teacher_secondary', 
    'school_enrol_primary_pct', 'school_enrol_secondary_pct', 'school_enrol_tertiary_pct'
]

for col in numerical_cols:
    df_clean[col] = df_clean.groupby('country')[col].ffill().bfill()

# Step C: Fill any remaining missing values that couldn't be interpolated with the global median
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        global_median = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(global_median)

# Step D: Round numerical values to 2 decimal places for clean D3.js rendering
df_clean[numerical_cols] = df_clean[numerical_cols].round(2)

# Step E: Final check to ensure 0 missing values remain
print("--- MISSING VALUES AFTER CLEANING ---")
print(df_clean.isnull().sum())

# Step F: Export the clean dataset to a new CSV file for your D3.js dashboard
df_clean.to_csv("clean_world_education_data.csv", index=False)
print("\nSuccess! Cleaned dataset saved as 'clean_world_education_data.csv'")

--- MISSING VALUES AFTER CLEANING ---
country                       0
country_code                  0
year                          0
gov_exp_pct_gdp               0
lit_rate_adult_pct            0
pri_comp_rate_pct             0
pupil_teacher_primary         0
pupil_teacher_secondary       0
school_enrol_primary_pct      0
school_enrol_secondary_pct    0
school_enrol_tertiary_pct     0
dtype: int64

Success! Cleaned dataset saved as 'clean_world_education_data.csv'
